# IAN 603 · Module 9
# Data Cleaning in PostgreSQL

Profile, standardize, validate, deduplicate, quarantine, and safely clean data without hiding uncertainty.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module9.html


## Learning goals

Work through the examples in order. Focus on what each SQL pattern does, what one output row represents, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database
db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script
print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**. You may save the connection settings on your own computer for the next run.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 9

Cleaning is **evidence-based decision making**.

A useful pipeline is:

```text
profile
  ↓
standardize representation
  ↓
validate
  ↓
flag uncertainty
  ↓
deduplicate carefully
  ↓
review / quarantine
  ↓
apply safe updates
```

### Do not collapse different states

| Raw state | Possible meaning |
|---|---|
| NULL | missing / unknown |
| blank text | present but empty |
| sentinel value | placeholder such as 0000000000 |
| invalid value | does not satisfy a rule |
| duplicate candidate | may represent the same entity |

The goal is not to make everything look clean. The goal is to create a defensible analytical representation while preserving uncertainty.


# Teaching database

Run the next cell to create or reset the Module 9 teaching data. Re-run it whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 9\n-- Data Cleaning in PostgreSQL\n-- ============================================================\n-- Companion file for s_module9.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- profiling, NULL/blank/sentinel values, standardization,\n-- validation, regex checks, quality flags, duplicate detection,\n-- survivor rules, quarantine/review, and safe updates.\n--\n-- Main principle: cleaning is evidence-based decision making.\n-- Standardize representation when justified; do not invent facts.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP VIEW IF EXISTS customers_clean_preview CASCADE;\nDROP VIEW IF EXISTS customer_quality_issues CASCADE;\nDROP TABLE IF EXISTS validated_contacts CASCADE;\nDROP TABLE IF EXISTS messy_customers CASCADE;\nDROP TABLE IF EXISTS orders CASCADE;\nDROP TABLE IF EXISTS products CASCADE;\nDROP TABLE IF EXISTS customers CASCADE;\n\nCREATE TABLE customers (\n    id integer PRIMARY KEY,\n    name text,\n    email text,\n    phone text,\n    address text,\n    signup_date date\n);\n\nCREATE TABLE products (\n    id integer PRIMARY KEY,\n    product_name text,\n    category text,\n    description text,\n    price numeric(10,2),\n    stock_quantity integer\n);\n\nCREATE TABLE orders (\n    id integer PRIMARY KEY,\n    customer_id integer REFERENCES customers(id),\n    order_date date NOT NULL,\n    total_amount numeric(10,2) CHECK (total_amount >= 0),\n    status text\n);\n\nCREATE TABLE messy_customers (\n    id integer PRIMARY KEY,\n    full_name text,\n    email text,\n    phone text,\n    signup_date_raw text\n);\n\nINSERT INTO customers VALUES\n(1,'John Doe','john.doe@email.com','(336) 555-1201','123 Main St',DATE '2023-01-15'),\n(2,'  JANE   SMITH  ',' JANE.SMITH@EMAIL.COM ','336.555.1202','456 Oak Ave',DATE '2023-02-20'),\n(3,'Bob Johnson',NULL,'3365551203',NULL,DATE '2023-03-10'),\n(4,'Alice Brown','alice.brown@test.com',NULL,'789 Pine Rd',DATE '2023-04-05'),\n(5,'CAROL WILSON','CAROL@TEST.COM','(336) 555-1205','101 Elm St',DATE '2023-05-12'),\n(6,'  David Lee  ','david.lee@example.com','336-555-1206','202 Maple Dr',DATE '2023-06-18'),\n(7,NULL,'empty.name@test.com','3365551207','303 Birch Ln',DATE '2023-07-22'),\n(8,'Duplicate User','duplicate@test.com','3365551208','404 Spruce St',DATE '2023-08-30'),\n(9,' Duplicate   User ',' DUPLICATE@TEST.COM ','(336) 555-1208',NULL,DATE '2023-09-01'),\n(10,'Test User','invalid-email','123','','2023-09-15'),\n(11,'Sentinel Phone','sentinel@example.com','0000000000','505 Cedar St',DATE '2023-10-01'),\n(12,'Blank Contact',' ',' ','',NULL);\n\nINSERT INTO products VALUES\n(1,'Laptop Computer','Electronics','High-performance laptop with 16GB RAM',999.99,10),\n(2,'Wireless Mouse','electronics','Ergonomic wireless mouse',29.99,25),\n(3,'Keyboard',NULL,'Mechanical keyboard',79.99,15),\n(4,'Monitor','Electronics','27 inch 4K monitor',-399.99,8),\n(5,'Headphones','AUDIO','Noise-cancelling headphones',199.99,NULL),\n(6,'','Electronics','Tablet device',299.99,5),\n(7,'Desk Lamp','Home Office','LED desk lamp',49.99,30),\n(8,'Notebook','stationery','Spiral notebook',NULL,100),\n(9,'Pen Set','Stationery','Set of 5 pens',12.99,50);\n\nINSERT INTO orders VALUES\n(101,1,DATE '2026-01-05',150.00,'shipped'),\n(102,2,DATE '2026-01-20',300.00,'pending'),\n(103,1,DATE '2026-02-10',200.00,'delivered'),\n(104,3,DATE '2026-03-04',450.00,'shipped'),\n(105,2,DATE '2026-03-18',125.00,'cancelled');\n\nINSERT INTO messy_customers VALUES\n(1,'  JOHN DOE  ','JOHN@EXAMPLE.COM','(555) 123-4567','2023-01-15'),\n(2,'jane smith','jane.smith@email.com','555.987.6543','01/20/2023'),\n(3,'bob johnson',NULL,'5551112233','2023-02-28'),\n(4,'Alice Brown','alice.brown@test.com',NULL,'03/15/2023'),\n(5,'  carol  wilson  ','CAROL@TEST.COM','(555) 444-5555','invalid_date'),\n(6,'David Lee','david.lee@example.com','555-999-8888','2023-04-01'),\n(7,'  ','empty.name@test.com','5550001111','2023-05-10'),\n(8,'Duplicate User','duplicate@test.com','5551112222','2023-06-15'),\n(9,' Duplicate   User ',' DUPLICATE@TEST.COM ','(555)111-2222','2023-06-15');"

execute_script(setup_sql)


# MODULE EXAMPLES


## Inspect customer source rows

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, name, email, phone, address, signup_date\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Inspect customer source rows")


## Missingness profile

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT COUNT(*) AS total_rows,\n       COUNT(name) AS nonnull_name,\n       COUNT(NULLIF(BTRIM(name),'')) AS nonblank_name,\n       COUNT(email) AS nonnull_email,\n       COUNT(NULLIF(BTRIM(email),'')) AS nonblank_email,\n       COUNT(phone) AS nonnull_phone,\n       COUNT(NULLIF(BTRIM(phone),'')) AS nonblank_phone\nFROM customers;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Missingness profile")


## Classify phone states

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, phone,\n       CASE\n         WHEN phone IS NULL THEN 'NULL'\n         WHEN BTRIM(phone) = '' THEN 'blank'\n         WHEN REGEXP_REPLACE(phone, '[^0-9]', '', 'g') IN ('0000000000','9999999999') THEN 'sentinel'\n         ELSE 'candidate value'\n       END AS phone_state\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Classify phone states")


## Normalize blank addresses

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, address,\n       NULLIF(BTRIM(address),'') AS normalized_address\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Normalize blank addresses")


## Preferred contact fallback

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, name,\n       COALESCE(NULLIF(BTRIM(email),''), NULLIF(BTRIM(phone),''), 'no contact method') AS preferred_contact\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Preferred contact fallback")


## Standardize customer names

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, name AS raw_name,\n       INITCAP(REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g')) AS clean_name\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Standardize customer names")


## Standardize customer emails

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, email AS raw_email,\n       LOWER(NULLIF(BTRIM(email),'')) AS clean_email\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Standardize customer emails")


## Standardize phone digits

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, phone AS raw_phone,\n       NULLIF(REGEXP_REPLACE(COALESCE(phone,''), '[^0-9]', '', 'g'),'') AS digits_only,\n       CASE\n         WHEN LENGTH(REGEXP_REPLACE(COALESCE(phone,''), '[^0-9]', '', 'g')) = 10\n         THEN REGEXP_REPLACE(COALESCE(phone,''), '[^0-9]', '', 'g')\n         ELSE NULL\n       END AS valid_10_digit_phone\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Standardize phone digits")


## Map product categories

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, product_name, category AS raw_category,\n       CASE\n         WHEN NULLIF(BTRIM(category),'') IS NULL THEN NULL\n         WHEN LOWER(BTRIM(category)) IN ('electronics','electronic','tech') THEN 'Electronics'\n         WHEN LOWER(BTRIM(category)) IN ('audio','headphones') THEN 'Audio'\n         WHEN LOWER(BTRIM(category)) IN ('stationery','office supplies') THEN 'Stationery'\n         WHEN LOWER(BTRIM(category)) IN ('home office','home & garden') THEN 'Home Office'\n         ELSE INITCAP(BTRIM(category))\n       END AS standard_category\nFROM products\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Map product categories")


## Flag product prices

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, product_name, price,\n       CASE\n         WHEN price IS NULL THEN 'missing'\n         WHEN price <= 0 THEN 'invalid - review'\n         ELSE 'valid candidate'\n       END AS price_status\nFROM products\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Flag product prices")


## Email validation

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, email,\n       CASE\n         WHEN NULLIF(BTRIM(email),'') IS NULL THEN 'missing'\n         WHEN LOWER(BTRIM(email)) ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN 'valid shape'\n         ELSE 'review'\n       END AS email_status\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Email validation")


## Phone validation

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, phone,\n       REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g') AS digits,\n       CASE WHEN LENGTH(REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g')) = 10 THEN 'valid shape'\n            WHEN NULLIF(BTRIM(phone),'') IS NULL THEN 'missing'\n            ELSE 'review' END AS phone_status\nFROM customers\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Phone validation")


## Product validation

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, product_name, category, price, stock_quantity,\n       CASE\n         WHEN NULLIF(BTRIM(product_name),'') IS NULL THEN 'missing product name'\n         WHEN price IS NULL THEN 'missing price'\n         WHEN price <= 0 THEN 'invalid price'\n         WHEN stock_quantity IS NOT NULL AND stock_quantity < 0 THEN 'negative stock'\n         ELSE 'passes selected rules'\n       END AS validation_status\nFROM products\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Product validation")


## Quality issue report

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH issues AS (\n  SELECT id, 'customer' AS entity, 'invalid_email' AS issue\n  FROM customers WHERE NULLIF(BTRIM(email),'') IS NOT NULL AND LOWER(BTRIM(email)) !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$'\n  UNION ALL\n  SELECT id, 'customer', 'invalid_phone'\n  FROM customers WHERE NULLIF(BTRIM(phone),'') IS NOT NULL AND LENGTH(REGEXP_REPLACE(phone,'[^0-9]','','g')) <> 10\n  UNION ALL\n  SELECT id, 'product', 'invalid_price'\n  FROM products WHERE price IS NULL OR price <= 0\n)\nSELECT entity, issue, COUNT(*) AS issue_count\nFROM issues\nGROUP BY entity, issue\nORDER BY issue_count DESC, entity, issue;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Quality issue report")


## Illustrative customer quality score

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT id, name,\n       (CASE WHEN NULLIF(BTRIM(name),'') IS NOT NULL THEN 25 ELSE 0 END +\n        CASE WHEN LOWER(COALESCE(BTRIM(email),'')) ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN 25 ELSE 0 END +\n        CASE WHEN LENGTH(REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g')) = 10 THEN 25 ELSE 0 END +\n        CASE WHEN signup_date IS NOT NULL AND signup_date <= DATE '2026-08-12' THEN 25 ELSE 0 END) AS illustrative_quality_score\nFROM customers\nORDER BY illustrative_quality_score, id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Illustrative customer quality score")


## Quality band summary

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH scored AS (\n SELECT id,\n       (CASE WHEN NULLIF(BTRIM(name),'') IS NOT NULL THEN 25 ELSE 0 END +\n        CASE WHEN LOWER(COALESCE(BTRIM(email),'')) ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN 25 ELSE 0 END +\n        CASE WHEN LENGTH(REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g')) = 10 THEN 25 ELSE 0 END +\n        CASE WHEN signup_date IS NOT NULL AND signup_date <= DATE '2026-08-12' THEN 25 ELSE 0 END) AS score\n FROM customers\n)\nSELECT CASE WHEN score=100 THEN 'Excellent'\n            WHEN score>=75 THEN 'Good'\n            WHEN score>=50 THEN 'Fair'\n            ELSE 'Poor' END AS quality_band,\n       COUNT(*) AS customer_count\nFROM scored\nGROUP BY 1\nORDER BY MIN(score) DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Quality band summary")


## Detect normalized duplicate groups

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT LOWER(BTRIM(email)) AS email_key,\n       INITCAP(REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g')) AS name_key,\n       COUNT(*) AS row_count,\n       ARRAY_AGG(id ORDER BY id) AS source_ids\nFROM customers\nWHERE NULLIF(BTRIM(email),'') IS NOT NULL\nGROUP BY LOWER(BTRIM(email)), INITCAP(REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g'))\nHAVING COUNT(*) > 1;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Detect normalized duplicate groups")


## Rank duplicate candidates

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH scored AS (\n  SELECT c.*,\n         (CASE WHEN NULLIF(BTRIM(name),'') IS NOT NULL THEN 1 ELSE 0 END +\n          CASE WHEN NULLIF(BTRIM(email),'') IS NOT NULL THEN 1 ELSE 0 END +\n          CASE WHEN NULLIF(BTRIM(phone),'') IS NOT NULL THEN 1 ELSE 0 END +\n          CASE WHEN NULLIF(BTRIM(address),'') IS NOT NULL THEN 1 ELSE 0 END) AS completeness_score\n  FROM customers AS c\n), ranked AS (\n  SELECT *, ROW_NUMBER() OVER (\n      PARTITION BY LOWER(BTRIM(email)), INITCAP(REGEXP_REPLACE(BTRIM(name),'[[:space:]]+',' ','g'))\n      ORDER BY completeness_score DESC, signup_date DESC NULLS LAST, id\n  ) AS survivor_rank\n  FROM scored\n)\nSELECT id, name, email, completeness_score, survivor_rank\nFROM ranked\nWHERE LOWER(BTRIM(email)) = 'duplicate@test.com'\nORDER BY survivor_rank;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Rank duplicate candidates")


## Possible duplicates by phone

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT a.id AS left_id, b.id AS right_id,\n       a.name AS left_name, b.name AS right_name,\n       a.email AS left_email, b.email AS right_email\nFROM customers AS a\nJOIN customers AS b ON a.id < b.id\nWHERE LOWER(REGEXP_REPLACE(COALESCE(a.phone,''),'[^0-9]','','g')) = LOWER(REGEXP_REPLACE(COALESCE(b.phone,''),'[^0-9]','','g'))\n  AND NULLIF(REGEXP_REPLACE(COALESCE(a.phone,''),'[^0-9]','','g'),'') IS NOT NULL\nORDER BY a.id,b.id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Possible duplicates by phone")


## Customer quality issue view

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "CREATE OR REPLACE VIEW customer_quality_issues AS\nSELECT id, name, email, phone,\n       CASE WHEN NULLIF(BTRIM(name),'') IS NULL THEN true ELSE false END AS missing_name,\n       CASE WHEN NULLIF(BTRIM(email),'') IS NOT NULL AND LOWER(BTRIM(email)) !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN true ELSE false END AS invalid_email,\n       CASE WHEN NULLIF(BTRIM(phone),'') IS NOT NULL AND LENGTH(REGEXP_REPLACE(phone,'[^0-9]','','g')) <> 10 THEN true ELSE false END AS invalid_phone\nFROM customers;\nSELECT * FROM customer_quality_issues\nWHERE missing_name OR invalid_email OR invalid_phone\nORDER BY id;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Create customers_clean_preview

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "CREATE OR REPLACE VIEW customers_clean_preview AS\nSELECT id,\n       INITCAP(REGEXP_REPLACE(BTRIM(name),'[[:space:]]+',' ','g')) AS clean_name,\n       LOWER(NULLIF(BTRIM(email),'')) AS clean_email,\n       CASE WHEN LENGTH(REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g'))=10\n            THEN REGEXP_REPLACE(phone,'[^0-9]','','g') ELSE NULL END AS clean_phone,\n       NULLIF(BTRIM(address),'') AS clean_address,\n       signup_date\nFROM customers;\nSELECT * FROM customers_clean_preview ORDER BY id;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Validated contacts with constraints

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "DROP TABLE IF EXISTS validated_contacts;\nCREATE TABLE validated_contacts (\n  contact_id integer PRIMARY KEY,\n  email text NOT NULL,\n  phone text,\n  CONSTRAINT email_shape_chk CHECK (email ~ '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'),\n  CONSTRAINT phone_digits_chk CHECK (phone IS NULL OR phone ~ '^[0-9]{10}$'),\n  CONSTRAINT email_unique UNIQUE (email)\n);\nINSERT INTO validated_contacts VALUES (1,'student@example.com','3365551212');\nSELECT * FROM validated_contacts;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Transaction + UPDATE RETURNING

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "BEGIN;\nUPDATE customers\nSET email = LOWER(BTRIM(email))\nWHERE email IS NOT NULL\n  AND email IS DISTINCT FROM LOWER(BTRIM(email))\nRETURNING id, email;\nROLLBACK;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Project: full cleaning pipeline

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "WITH normalized AS (\n SELECT id,\n        NULLIF(REGEXP_REPLACE(BTRIM(full_name),'[[:space:]]+',' ','g'),'') AS name_norm,\n        LOWER(NULLIF(BTRIM(email),'')) AS email_norm,\n        NULLIF(REGEXP_REPLACE(COALESCE(phone,''),'[^0-9]','','g'),'') AS phone_digits,\n        signup_date_raw\n FROM messy_customers\n), typed AS (\n SELECT *,\n        CASE WHEN signup_date_raw ~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN signup_date_raw::date\n             WHEN signup_date_raw ~ '^[0-9]{2}/[0-9]{2}/[0-9]{4}$' THEN TO_DATE(signup_date_raw,'MM/DD/YYYY')\n             ELSE NULL END AS signup_date_clean\n FROM normalized\n), ranked AS (\n SELECT *, ROW_NUMBER() OVER (\n   PARTITION BY email_norm, INITCAP(name_norm)\n   ORDER BY (CASE WHEN name_norm IS NOT NULL THEN 1 ELSE 0 END + CASE WHEN email_norm IS NOT NULL THEN 1 ELSE 0 END + CASE WHEN LENGTH(phone_digits)=10 THEN 1 ELSE 0 END + CASE WHEN signup_date_clean IS NOT NULL THEN 1 ELSE 0 END) DESC, id\n ) AS duplicate_rank\n FROM typed\n)\nSELECT id, INITCAP(name_norm) AS clean_name, email_norm AS clean_email,\n       CASE WHEN LENGTH(phone_digits)=10 THEN phone_digits ELSE NULL END AS clean_phone,\n       signup_date_clean,\n       duplicate_rank,\n       CASE WHEN name_norm IS NULL OR email_norm IS NULL OR signup_date_clean IS NULL OR (phone_digits IS NOT NULL AND LENGTH(phone_digits)<>10) THEN 'review' ELSE 'candidate clean row' END AS row_status\nFROM ranked\nORDER BY duplicate_rank, id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Project: full cleaning pipeline")


## Assessment starter

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- Assessment: return one row per customer with cleaned name/email/phone,\n-- a row-level quality status, and a duplicate rank. Do not overwrite the source table.\nWITH normalized AS (\n  SELECT id, name, email, phone, address, signup_date\n  FROM customers\n)\nSELECT\n  -- build the cleaned analytical output here\nFROM normalized;"

display_query(query, title="Assessment starter")


# BEFORE/AFTER VALIDATION


Profile missing and blank contact fields.
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (WHERE name IS NULL) AS null_names,
    COUNT(*) FILTER (WHERE NULLIF(BTRIM(name),'') IS NULL) AS unusable_names,
    COUNT(*) FILTER (WHERE NULLIF(BTRIM(email),'') IS NULL) AS unusable_emails,
    COUNT(*) FILTER (WHERE NULLIF(BTRIM(phone),'') IS NULL) AS unusable_phones
FROM customers;
Detect duplicate normalized email values.
SELECT
    LOWER(BTRIM(email)) AS normalized_email,
    COUNT(*) AS row_count,
    ARRAY_AGG(id ORDER BY id) AS source_ids
FROM customers
WHERE NULLIF(BTRIM(email),'') IS NOT NULL
GROUP BY LOWER(BTRIM(email))
HAVING COUNT(*) > 1
ORDER BY normalized_email;
Keep invalid rows visible for review rather than silently fixing.
SELECT id, name, email, phone
FROM customers
WHERE NULLIF(BTRIM(name),'') IS NULL
   OR (NULLIF(BTRIM(email),'') IS NOT NULL
       AND LOWER(BTRIM(email)) !~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$')
   OR (NULLIF(BTRIM(phone),'') IS NOT NULL
       AND LENGTH(REGEXP_REPLACE(phone,'[^0-9]','','g')) <> 10)
ORDER BY id;


# SAFE CLEANING WORKFLOW


1. PROFILE raw values.
2. PRESERVE source evidence.
3. NORMALIZE representation.
4. VALIDATE against documented rules.
5. RESOLVE duplicates/invalid records with explicit policy.
6. PREVENT recurrence with constraints or upstream controls.


# Practice

1. Standardize customer names and emails.
2. Normalize phone numbers to digits-only candidates.
3. Validate product price and stock rules.
4. Build a quality-issue summary by issue type.
5. Detect duplicates using normalized keys.
6. Explain which records require human review instead of repair.
CHECKLIST
[ ] I profile before changing values.
[ ] I distinguish NULL, blank, sentinel, and invalid values.
[ ] I preserve raw evidence when cleaning.
[ ] I document deduplication/survivor rules.
[ ] I validate results after cleaning.

Work on each task before reviewing earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# display_query(practice_query, title="Practice")


# Module 9 summary

Before moving on, make sure you can explain the main patterns in your own words.

For any query, ask:
1. What question does it answer?
2. What does one input row represent?
3. What does one output row represent?
4. What SQL rule changed or filtered the data?
5. How would you validate the result?

## Next
Continue to **Module 10**.
